# microsimflow on Google Colab

This notebook provides an automated environment to run `microsimflow` experiments directly on Google Colab. 

**Hardware Requirement:**
Please ensure that your Colab runtime is set to use a **T4 GPU** (Go to `Runtime` > `Change runtime type` > Select `T4 GPU`).

**Notes on this Colab environment:**
* **chfem solver:** We are compiling a temporarily forked version of `chfem` to bypass a build error in the original repository. It is configured specifically for the T4 architecture (`-DCMAKE_CUDA_ARCHITECTURES=75`).
* **PuMA solver:** The `pumapy` module is not supported in this lightweight Colab setup. All scripts are configured to use `chfem` for homogenization.

### Step 1: Setup Workspace & Clone Repository
Clone the `microsimflow` scripts into a dedicated workspace directory.

In [ ]:
!git clone https://github.com/hikuram/microsimflow.git /content/microsimflow
!mkdir -p /content/workspace
!cp /content/microsimflow/*.py /content/workspace

### Step 2: Install Dependencies & Compile `chfem`
Verify the CUDA compiler, install necessary Python libraries like `pyvista`, and compile the GPU-accelerated `chfem` solver.

In [ ]:
# 1. Verify CUDA compiler
!nvcc --version

# 2. Clone and compile chfem (using the fork with the typo fix)
!git clone https://github.com/hikuram/chfem.git /content/chfem
!cd /content/chfem && mkdir -p build && cd build && cmake .. \
      -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_CUDA_ARCHITECTURES=75 \
      -DCMAKE_C_FLAGS="-DCUDAPCG_MATKEY_32BIT" \
      -DCMAKE_CUDA_FLAGS="-DCUDAPCG_MATKEY_32BIT" && \
    make -j2

# 3. Install Python dependencies
!pip install pyvista numba

import os
# Add compiled chfem to system PATH
os.environ['PATH'] = '/content/chfem:' + os.environ['PATH']

### Step 3: Run Experiments
Execute the pre-defined parameter sweep experiments. Each script generates structural data (`.vti`), visualization images (`.png`), and aggregates properties into a `.csv` file.

In [ ]:
# Experiment 0: Standard Percolation Sweep (Varying filler volume fraction and radius)
%cd /content/workspace
!python run_exp0_percolation.py
!python plot_exp0_percolation.py

In [ ]:
# Experiment 1: Agglomeration Sweep (Testing the effect of fiber entanglement/clustering)
%cd /content/workspace
!python run_exp1_agglom.py
!python plot_exp1_agglom.py

In [ ]:
# Experiment 2: Gyroid Percolation (Fibers embedded in a co-continuous polymer blend)
%cd /content/workspace
!python run_exp2_gyroid.py
!python plot_exp2_gyroid.py

In [ ]:
# Experiment 3: Hybrid Synergy Sweep (Mixing rigid fibers and flakes)
%cd /content/workspace
!python run_exp3_hybrid.py
!python plot_exp3_hybrid.py

In [ ]:
# Experiment 4: Scale & Box Size Check (Validating representative volume element limits)
%cd /content/workspace
!python run_exp4_scale_check.py

### Step 4: Run Randomized Test Suite
Generate and execute a batch of random configurations to verify pipeline robustness.

In [ ]:
%cd /content/workspace
!python generate_test.py
!python run_tests.py

### Step 5: Download All Results
Archive the generated data (CSVs, VTI 3D grids, and PNG slices) and download them to your local machine.

In [ ]:
%cd /content/workspace
!zip -r /content/microsimflow_results.zip *
from google.colab import files
files.download("/content/microsimflow_results.zip")